In [ ]:
from selenium import webdriver # Navegador
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException

import undetected_chromedriver as uc
from datetime import datetime
import time
import re


In [ ]:
place = "Fiumicino"

In [ ]:
get_dates = True
flexible_dates = True
start_date = '2025-05-24'
end_date = '2025-05-25'
mode = 'Weekend'
index = 2

In [ ]:
def select_dates(driver, start_date_str, end_date_str):
    def open_calendar():
        calendar_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-testid="searchbox-dates-container"]'))
        )
        calendar_btn.click()
        time.sleep(0.5)

    def get_visible_months():
        # Read all <h3> month headers
        months = driver.find_elements(By.CSS_SELECTOR, 'h3.e1eebb6a1e.ee7ec6b631')
        return [m.text.lower() for m in months if m.text.strip()]

    def go_to_month(target_date):
        # Create "agosto 2025" string
        months_it = [
            "gennaio", "febbraio", "marzo", "aprile", "maggio", "giugno",
            "luglio", "agosto", "settembre", "ottobre", "novembre", "dicembre"
        ]
        target_month_str = f"{months_it[target_date.month - 1]} {target_date.year}"
        print(f"Looking for: {target_month_str}")

        for i in range(12):
            visible = get_visible_months()
            print(f"[{i}] Visible months: {visible}")
            if any(target_month_str in m for m in visible):
                return
            next_btn = WebDriverWait(driver, 2).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label="mese successivo"]'))
            )
            next_btn.click()
            time.sleep(0.6)
        raise Exception(f"Could not find month: {target_month_str}")

    def click_date(date_str):
        elem = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, f'span[data-date="{date_str}"]'))
        )
        elem.click()

    # Parse strings to datetime
    start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = datetime.strptime(end_date_str, '%Y-%m-%d')

    # Run
    open_calendar()
    go_to_month(start_date)
    click_date(start_date_str)
    go_to_month(end_date)
    click_date(end_date_str)


In [ ]:
def dismiss_cookie_banner(driver):
    try:
        consent_btn = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        consent_btn.click()
        print("Cookie banner dismissed.")
        time.sleep(0.5)
    except TimeoutException:
        print("No cookie banner found.")


In [ ]:
def retrying_find_click(by, value):
    attempts = 0
    while attempts < 5:
        try:
            element = driver.find_element(by, value)
            element.click()
            break
        except StaleElementReferenceException:
            attempts += 1

In [ ]:
def scroll_to_load(driver, scrolls=10, pause=1.5):
    """
    Scrolls down the page a few times to trigger initial lazy load.
    """
    print(f"📜 Scrolling {scrolls} times to trigger lazy loading...")
    for i in range(scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        
def load_all_results(driver, max_clicks=20):
    for i in range(max_clicks):
        try:
            print(f"[{i}] Checking for 'Carica più risultati' button...")
            load_more_btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, '//button[.//span[contains(text(), "Carica più risultati")]]'))
            )
            load_more_btn.click()
            print(f"[{i}] Clicked. Waiting for more results to load...")
            time.sleep(2)  # Adjust as needed depending on connection
        except TimeoutException:
            print("No more 'Carica più risultati' button found.")
            break
        except ElementClickInterceptedException:
            print("Button found but not clickable — retrying after scrolling...")
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
            time.sleep(1)
            load_more_btn.click()
            
            
def get_all_prices(driver):
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'span[data-testid="price-and-discounted-price"]'))
    )
    price_elems = driver.find_elements(By.CSS_SELECTOR, 'span[data-testid="price-and-discounted-price"]')
    prices = [elem.text.strip() for elem in price_elems if elem.text.strip()]
    numeric_prices = [int(re.sub(r"[^\d]", "", p)) for p in prices]

    return numeric_prices

In [ ]:
def extract_listing_info(driver, get_dates):
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="property-card"]'))
    )
    listings = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="property-card"]')
    
    results = []
    print(len(listings))
    for i, listing in enumerate(listings):
        if get_dates:
            try:
                price = listing.find_element(By.CSS_SELECTOR, 'span[data-testid="price-and-discounted-price"]').text.strip()
            except:
                price = None
        else:
            price=None
        try:
            location = listing.find_element(By.CSS_SELECTOR, 'span[data-testid="address"]').text.strip()
        except:
            location = None
        try:
            distance = listing.find_element(By.CSS_SELECTOR, 'span[data-testid="distance"]').text.strip()
        except:
            distance = None
        try:
            title = listing.find_element(By.CSS_SELECTOR, 'div[data-testid="title"]').text.strip()
        except:
            title = None
        try:
            link_element = listing.find_element(By.CSS_SELECTOR, 'a[data-testid="title-link"]')
            href = link_element.get_attribute("href")
        except:
            href = None
            
        results.append({
            "title": title,
            "link": href,
            "price": price,
            "location": location,
            "distance_from_center": distance
        })
        if i%10==0:
            print(f"Done iteration {i}/{len(listings)}")

    return results


In [ ]:
def click_filter_by_name(driver, filter_text, timeout=5):
    """
    Clicks a filter checkbox that exactly matches the provided visible label text.
    Ignores elements in the sort-by ("Ordina per") section.
    """
    try:
        print(f"🔍 Trying to click filter: '{filter_text}'")

        # XPath that finds exact text inside filters (case-sensitive)
        xpath = (
            f'//div[@data-testid="filters-group-label-content" and normalize-space(text())="{filter_text}"]'
            f'/ancestor::label[1]'
        )

        label_elem = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.XPATH, xpath))
        )

        # 🛑 Check if the label is inside the sort section (like "Ordina per")
        parent_sort_section = label_elem.find_elements(By.XPATH, './ancestor::span[contains(@class, "cac967781c")]')
        if parent_sort_section:
            print(f"⚠️ Skipping '{filter_text}' because it belongs to sort section.")
            return False

        # ✅ Scroll and click
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", label_elem)
        label_elem.click()
        print(f"✅ Clicked filter: '{filter_text}'")
        return True

    except TimeoutException:
        print(f"❌ Filter '{filter_text}' not found in timeout.")
        return False
    except Exception as e:
        print(f"❌ Error while clicking filter '{filter_text}': {e}")
        return False


In [ ]:
# Mode = "Weekend", "Week", "Month" (default )
#Se index = 1 mese corrente, 2 = M+1 etc
def flexible_data(Mode, index): 
    #Click su Date Flessibili    
    try:
        x_path = "/html/body/div[3]/div[2]/div/form/div/div[2]/div/div/div/nav/div[1]/ul/li[2]/button"
        retrying_find_click(By.XPATH, x_path)
    except:
        button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, '[data-testid="searchbox-dates-container"]')))
        button.click()
        time.sleep(3)
        retrying_find_click(By.XPATH, x_path)
     
    if Mode == "Weekend":
        #Click su Weekend    
        x_path = "/html/body/div[3]/div[2]/div/form/div/div[2]/div/div/div/nav/div[3]/div/div[1]/div[1]/div/fieldset/div/div[1]/label/span[3]/div"
    else:
        if Mode == "Week":
            #Click su 1 settimana
            x_path = "/html/body/div[3]/div[2]/div/form/div/div[2]/div/div/div/nav/div[3]/div/div[1]/div[1]/div/fieldset/div/div[2]/label/span[2]"
        else:
            #Click su 1 mese
            x_path = "/html/body/div[3]/div[2]/div/form/div/div[2]/div/div/div/nav/div[3]/div/div[1]/div[1]/div/fieldset/div/div[3]/label/span[2]"
    
    retrying_find_click(By.XPATH, x_path)
    
    
    x_path = "/html/body/div[3]/div[2]/div/form/div/div[2]/div/div/div/nav/div[3]/div/div[1]/div[2]/div/fieldset/div/ul/li["+str(index)+"]/label/span/span/div"
    retrying_find_click(By.XPATH, x_path)
    
    x_path = "/html/body/div[3]/div[2]/div/form/div/div[2]/div/div/div/nav/div[3]/div/div[2]/button/span"
    retrying_find_click(By.XPATH, x_path)
    

In [ ]:
def get_selected_dates(driver, timeout=5):
    """
    Returns the selected dates or flexible period text from the Booking.com date picker field.
    Handles both fixed dates and flexible dates.
    """
    try:
        # Wait for the main container
        container = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'span.a8887b152e.c7ce171153'))
        )

        try:
            # Try to get fixed start and end dates
            start_date = container.find_element(By.CSS_SELECTOR, 'span[data-testid="date-display-field-start"]').text.strip()
            end_date = container.find_element(By.CSS_SELECTOR, 'span[data-testid="date-display-field-end"]').text.strip()
            print(f"📅 Selected fixed dates: {start_date} → {end_date}")
            return {
                "mode": "fixed",
                "start_date": start_date,
                "end_date": end_date
            }
        except:
            # If no start/end spans, fall back to generic flexible text
            flexible_text = container.text.strip()
            print(f"📅 Selected flexible period: {flexible_text}")
            return {
                "mode": "flexible",
                "text": flexible_text
            }

    except TimeoutException:
        print("⚠️ Date picker field not found.")
        return None


In [ ]:
# Setup Chrome options
options = uc.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
options.add_argument("--window-size=1920x1080")

# options.add_argument("--headless")  # Optional: hide the browser

# This automatically finds the correct version of ChromeDriver
driver = uc.Chrome(options=options)

# Add any options you need, e.g.:
# chrome_options.add_argument("--headless")

# Allow automatic downloads to a custom path
driver.command_executor._commands["send_command"] = ("POST", '/session/$sessionId/chromium/send_command')

# Set timeouts
driver.implicitly_wait(2)
driver.set_script_timeout(900)
driver.set_page_load_timeout(900)
#driver.maximize_window()


In [ ]:
driver.get("https://www.booking.com/")

In [ ]:
dismiss_cookie_banner(driver)
time.sleep(2)

In [ ]:
#Insert Location
location_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "/html/body/div[3]/div[2]/div/form/div/div[1]/div/div/div[1]/div/div/input")))
location_input.send_keys(place)

#Click sul primo elemento
retrying_find_click(By.CLASS_NAME, 'a3332d346a')
time.sleep(2)

In [ ]:
if get_dates:
    if not flexible_dates:
        select_dates(driver, start_date, end_date)
        time.sleep(2)
    else:
        flexible_data(mode, index)
        time.sleep(2)

In [ ]:
period = get_selected_dates(driver)

In [ ]:
# Click the Search button
retrying_find_click(By.XPATH, '/html/body/div[3]/div[2]/div/form/div/div[4]/button')
time.sleep(2)

In [ ]:
# Remove Genius banner
try:
    x_path = "/html/body/div[18]/div/div/div/div[1]/div[1]/div/button"
    retrying_find_click(By.XPATH, x_path)
except:
    pass

def close_login_popup(driver, timeout=3):
    """
    Clicks the 'Ignore login info' popup button if it appears.
    """
    try:
        close_btn = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.XPATH, '//button[@aria-label="Ignora le informazioni sull\'accesso."]'))
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", close_btn)
        close_btn.click()
        print("✅ Closed login popup.")
    except TimeoutException:
        print("ℹ️ Login popup not shown (nothing to close).")
        
close_login_popup(driver, 5)

In [ ]:
# Click multiple filters
for f in ["Appartamenti", "Case e interi appartamenti", "Fai da te: cucina/angolo cottura", "Bagno privato"]:
    click_filter_by_name(driver, f)
    time.sleep(2)

In [ ]:
scroll_to_load(driver, scrolls=10, pause=1.5)
load_all_results(driver, max_clicks=20)
time.sleep(2)

In [ ]:
p = extract_listing_info(driver, get_dates)
time.sleep(2)


In [ ]:
driver.quit()

In [ ]:
import pandas as pd
df = pd.DataFrame(p)

In [ ]:
df_filtered = df[df['location']==place]

In [ ]:
period

In [ ]:
if get_dates:
    if period['mode']=='fixed':
        name = f"{place}_{period['start_date']}_{period['end_date']}.xlsx"
    else:
        name = f"{place}_{period['text']}.xlsx"
else:
    name = f"{place}.xlsx"


In [ ]:
with pd.ExcelWriter(name, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name='All', index=False)
    df_filtered.to_excel(writer, sheet_name='Filtered', index=False)

In [ ]:
!pip install pydrive2